In [1]:
import requests
from dotenv import load_dotenv
import os
import json
from PyPDF2 import PdfReader
from openai import OpenAI
import gradio as gr

In [2]:
load_dotenv(override=True)

True

In [3]:
pushover_user=os.getenv('PUSHOVER_USER')
pushover_token=os.getenv('PUSHOVER_TOKEN')

In [4]:
def push(message):
    data_sending={'token':pushover_token,'user':pushover_user,'message':message}
    requests.post(url='https://api.pushover.net/1/messages.json',data=data_sending)

In [5]:
def record_user_info(email,name="Not provided",phone_number="Not provided"):
    push(f"Record user info of {name} with email: {email} and phone number: {phone_number}")
    return {"recorded":"sucessfully"}

In [6]:
def record_unanswered_question(question):
    push(f"This question was not answered :{question}")
    return {"recorded":"sucessfully"}

In [7]:
record_user_info_json={
    "name":"record_user_info",
    "description":"Record user info like name, email, phone number. Use this function if user want to keep in touch",
    "parameters":{
        "type":"object",
        "properties":{
            "email":{
                "type":"string",
                "description":"The email address of user"
            },
            "name":{
                "type":"string",
                "description":"Name of user"
            },
            "phone_number":{
                "type":"string",
                "description":"Phone number of user"
            }
        },
        "required":[
            "email"
        ],
        "additionalProperties":False
    }
}

In [8]:
record_unanswered_question_json={
    "name":"record_unanswered_question",
    "description":"Record the question that cannot anwser. Use this function when cannot answer the question from user",
    "parameters":{
        "type":"object",
        "properties":{
            "question":{
                "type":"string",
                "description":"The question that couldn't be answered"
            },
        },
        "required":[
            "question"
        ],
        "additionalProperties":False
    }
}

In [9]:
tool=[
    {"type":"function","function":record_user_info_json},
    {"type":"function","function":record_unanswered_question_json}
]


In [10]:
def handle_tool_calls(tool_calls):
    results=[]
    for tool in tool_calls:
        tool_id=tool.id
        tool_name=tool.function.name
        tool_arguments=json.loads(tool.function.arguments)
        function=globals().get(tool_name)
        result=function(**tool_arguments)
        results.append({"role":"tool",
                "content":json.dumps(result),
                "tool_call_id":tool_id})
    return results

In [11]:
file=PdfReader('./quan/Quan Nguyen.pdf')
linkedin=''
for page in file.pages:
    text=page.extract_text()
    if text:
        linkedin+=text

In [12]:
with open('./quan/Summary.txt') as file:
    summary=file.read()

In [13]:
name="Quan Nguyen"

In [14]:
system_prompt=f"""You are acting as {name} to answer question about him in most professional way
You will be provided CV and summary to answer to the users' question, please be faithful as possible. If you dont know the answer, just say you dont know
Here is the CV: {linkedin}
Here is the summary: {summary}
With this context please chat with user, always stay in {name} character 
"""

In [15]:
openai=OpenAI()

In [16]:
def chat(message,history):
    messages=[{"role":"system","content":system_prompt}]+history+[{"role":"user","content":message}]
    done=False
    while not done:
        response=openai.chat.completions.create(model="gpt-4o-mini",messages=messages,tools=tool)
        finish_reason=response.choices[0].finish_reason
        if finish_reason=="tool_calls":
            message_=response.choices[0].message
            tool_calls=message_.tool_calls
            result=handle_tool_calls(tool_calls)
            messages.append(message_)
            messages.extend(result)
        else:
            done=True
    return response.choices[0].message.content

In [17]:
gr.ChatInterface(chat,type="messages").launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
